# ResNet on Google Colab

This notebook runs **ResNet-50 only** on **Google Colab GPU** for the FERreal project. It uses the repo's existing training code and only trains the ResNet model.

## 1. Setup GPU Environment on Google Colab

1. Open this notebook in Colab.
2. Go to Runtime → Change runtime type.
3. Set Hardware accelerator to GPU.
4. Run the next cell to verify that PyTorch can see the GPU.

This notebook uses the FERreal repository and trains **ResNet-50 only**.

In [ ]:
import torch

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Clone the FERreal Repository and Install Dependencies

Run this once per Colab session.

In [ ]:
from pathlib import Path
import os

repo_dir = Path('/content/FERreal')
if not repo_dir.exists():
    !git clone https://github.com/aathithc/FERreal.git /content/FERreal
%cd /content/FERreal

!pip -q install -r requirements.txt

## 3. Import Required Libraries

These imports use the FERreal project code directly.

In [ ]:
import json
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import DataLoader

from src.models import MODEL_REGISTRY
from src.data.dataset import FER2013Dataset
from src.data.transforms import get_train_transforms, get_eval_transforms
from src.training.utils import set_seed, compute_class_weights, save_checkpoint
from src.evaluation.evaluate import evaluate_model, print_results

## 4. Load and Preprocess Data

This notebook uses the FER-2013 dataset. Upload your Kaggle API file as `kaggle.json` or set the Kaggle credentials in Colab secrets before downloading.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

# Upload kaggle.json if it is not already present
kaggle_dir = Path('/root/.kaggle')
kaggle_dir.mkdir(parents=True, exist_ok=True)

if not (kaggle_dir / 'kaggle.json').exists():
    print('Upload your kaggle.json file now')
    uploaded = files.upload()
    if 'kaggle.json' in uploaded:
        shutil.move('kaggle.json', str(kaggle_dir / 'kaggle.json'))

!chmod 600 /root/.kaggle/kaggle.json

# Download and extract dataset if needed
if not Path('/content/FERreal/data/fer2013').exists():
    !bash scripts/download_data.sh

print('Dataset ready:', Path('/content/FERreal/data/fer2013').exists())

## 5. Build ResNet Model

This uses the repository's ResNet implementation and switches the config to 224×224 RGB input for Colab GPU training.

In [ ]:
set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

with open('configs/default.yaml') as f:
    cfg = yaml.safe_load(f)

# ResNet-specific settings for Colab GPU
cfg['image_size'] = 224
cfg['channels'] = 3
cfg['batch_size'] = 64
cfg['epochs'] = 20
cfg['early_stopping_patience'] = 5

train_tf = get_train_transforms(cfg['image_size'], cfg['channels'])
eval_tf = get_eval_transforms(cfg['image_size'], cfg['channels'])

train_ds = FER2013Dataset(cfg['data_root'], split='train', transform=train_tf)
val_ds = FER2013Dataset(cfg['data_root'], split='test', transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=cfg['batch_size'], shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=cfg['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

class_names = train_ds.classes
print('Classes:', class_names)

model = MODEL_REGISTRY['resnet'](num_classes=len(class_names)).to(device)
print(model.__class__.__name__)

weights = compute_class_weights(train_ds, device)
criterion = torch.nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg.get('weight_decay', 1e-4))

## 6. Train the Model

This trains ResNet-50 on the GPU and saves the best checkpoint to the notebook runtime (or Google Drive if you mount it).

In [ ]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['epochs'])
best_val_acc = 0.0
best_path = Path('checkpoints') / 'resnet' / 'best.pt'
best_path.parent.mkdir(parents=True, exist_ok=True)

for epoch in range(1, cfg['epochs'] + 1):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += images.size(0)

    scheduler.step()

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += images.size(0)

    train_acc = train_correct / train_total
    val_acc = val_correct / val_total
    print(f'Epoch {epoch:03d}/{cfg["epochs"]} | train loss {train_loss/train_total:.4f} acc {train_acc*100:.2f}% | val loss {val_loss/val_total:.4f} acc {val_acc*100:.2f}%')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_checkpoint({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_acc': best_val_acc,
            'class_names': class_names,
            'cfg': cfg,
        }, best_path)
        print(f'  -> saved best checkpoint to {best_path}')

print(f'Best validation accuracy: {best_val_acc*100:.2f}%')

## 7. Evaluate Model Performance

This loads the best checkpoint and reports accuracy, per-class F1, and a confusion matrix.

In [ ]:
best_ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(best_ckpt['model_state_dict'])

results = evaluate_model(model, val_loader, device, class_names)
print_results(results)

cm = results['confusion_matrix']
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('ResNet-50 Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()